# Dynamic Iceberg Tables Lab Guide

This notebook walks you through creating **Dynamic Iceberg Tables** that build a silver-layer aggregation pipeline over bronze balloon game event data.

Each Dynamic Table reads raw JSON events from the bronze Iceberg table, parses them, and materializes a specific aggregation — player leaderboards, color stats, real-time scores, and performance trends.

**Prerequisites:**
- A Catalog-Linked Database (CLD) reading the bronze Iceberg table
- USAGE privilege on the warehouse and external volume
- Role permissions to create Dynamic Tables and read the bronze source

---

## Step 1: Configure Your Environment

Set the variables below to match your environment. All subsequent SQL cells reference these variables via Jinja templating — change them once and everything updates.

> ### 🛑 STOP — Update the variables above before proceeding!
>
> * Make sure you have set **all five variables** (`WAREHOUSE`, `EXTERNAL_VOLUME`, `TARGET_LAG`, `BRONZE_ICEBERG_TABLE`, `DB_NAME`) to match your environment, then **run the cell above** before continuing. All subsequent SQL cells depend on these values.
> * The External Volume and related AWS resources can be created using `task dt:extvol-create` from withing the [Lab Sources](https://github.com/Snowflake-Labs/sfguide-lakehouse-iceberg-production-pipelines)

In [ ]:
WAREHOUSE = 'KAMESH_DEMOS_S'
EXTERNAL_VOLUME = 'KSAMPATH_BALLOON_SILVER_EXTERNAL_VOLUME'
TARGET_LAG = '5 minutes'
BRONZE_ICEBERG_TABLE = 'balloon_game_events."ksampath_balloon_pops"."balloon_game_events"'
DB_NAME = 'KSAMPATH_balloon_silver'

---

## Step 2: Create the Silver Database and Schema

Create a Snowflake-managed database and schema to hold the silver-layer Dynamic Iceberg Tables.

In [ ]:
%%sql -r dataframe_1
USE ROLE ACCOUNTADMIN;

In [ ]:
%%sql -r create_db_result
CREATE DATABASE IF NOT EXISTS {{DB_NAME}}
  COMMENT = 'Snowflake-managed silver (Dynamic Iceberg Tables over CLD bronze)';

In [ ]:
%%sql -r create_schema_result
CREATE SCHEMA IF NOT EXISTS {{DB_NAME}}.silver
  COMMENT = 'Aggregates from bronze balloon_game_events (JSON column event)';

---

## Step 2b: Set Notebook Context

Point the notebook session at the database and schema we just created so subsequent DDL runs in the right place.

In [ ]:
%%sql -r set_context_result
USE ROLE ACCOUNTADMIN;
USE DATABASE {{DB_NAME}};
USE SCHEMA silver;

---

## Step 3: Player Leaderboard (`dt_player_leaderboard`)

Aggregates each player's **total score**, **bonus pops** (favorite-color matches), and **last event timestamp**. This is the go-to table for ranking players.

In [ ]:
%%sql -r dt_leaderboard_result
CREATE OR REPLACE DYNAMIC ICEBERG TABLE {{DB_NAME}}.silver.dt_player_leaderboard (
  player STRING,
  total_score NUMBER(38,0),
  bonus_pops NUMBER(38,0),
  last_event_ts TIMESTAMP_NTZ
)
  TARGET_LAG = '{{TARGET_LAG}}'
  WAREHOUSE = {{WAREHOUSE}}
  EXTERNAL_VOLUME = '{{EXTERNAL_VOLUME}}'
  CATALOG = 'SNOWFLAKE'
  BASE_LOCATION = 'balloon_lab/dt_player_leaderboard'
AS
SELECT
  e.player AS player,
  SUM(e.score_i) AS total_score,
  COUNT_IF(e.fav_bonus) AS bonus_pops,
  MAX(e.ts) AS last_event_ts
FROM (
  SELECT
    v:player::STRING AS player,
    v:balloon_color::STRING AS balloon_color,
    v:score::INTEGER AS score_i,
    v:favorite_color_bonus::BOOLEAN AS fav_bonus,
    v:event_ts::TIMESTAMP_NTZ AS ts
  FROM (
    SELECT PARSE_JSON(event) AS v
    FROM {{BRONZE_ICEBERG_TABLE}}
  ) q
) e
GROUP BY e.player;

---

## Step 4: Balloon Color Stats (`dt_balloon_color_stats`)

Breaks down each player's performance **by balloon color** — pops, points, and bonus hits. Useful for analyzing color preferences and strategies.

In [ ]:
%%sql -r dt_color_stats_result
CREATE OR REPLACE DYNAMIC ICEBERG TABLE {{DB_NAME}}.silver.dt_balloon_color_stats (
  player STRING,
  balloon_color STRING,
  balloon_pops NUMBER(38,0),
  points_by_color NUMBER(38,0),
  bonus_hits NUMBER(38,0),
  last_event_ts TIMESTAMP_NTZ
)
  TARGET_LAG = '{{TARGET_LAG}}'
  WAREHOUSE = {{WAREHOUSE}}
  EXTERNAL_VOLUME = '{{EXTERNAL_VOLUME}}'
  CATALOG = 'SNOWFLAKE'
  BASE_LOCATION = 'balloon_lab/dt_balloon_color_stats'
AS
SELECT
  e.player,
  e.balloon_color,
  COUNT(*) AS balloon_pops,
  SUM(e.score_i) AS points_by_color,
  COUNT_IF(e.fav_bonus) AS bonus_hits,
  MAX(e.ts) AS last_event_ts
FROM (
  SELECT
    v:player::STRING AS player,
    v:balloon_color::STRING AS balloon_color,
    v:score::INTEGER AS score_i,
    v:favorite_color_bonus::BOOLEAN AS fav_bonus,
    v:event_ts::TIMESTAMP_NTZ AS ts
  FROM (
    SELECT PARSE_JSON(event) AS v
    FROM {{BRONZE_ICEBERG_TABLE}}
  ) q
) e
GROUP BY e.player, e.balloon_color;

---

## Step 5: Real-Time Scores (`dt_realtime_scores`)

Computes player scores in **15-second sliding windows** using `TIME_SLICE`. Shows how scores accumulate in near-real-time micro-batches.

In [ ]:
%%sql -r dt_realtime_result
CREATE OR REPLACE DYNAMIC ICEBERG TABLE {{DB_NAME}}.silver.dt_realtime_scores (
  player STRING,
  total_score NUMBER(38,0),
  window_start TIMESTAMP_NTZ,
  window_end TIMESTAMP_NTZ
)
  TARGET_LAG = '{{TARGET_LAG}}'
  WAREHOUSE = {{WAREHOUSE}}
  EXTERNAL_VOLUME = '{{EXTERNAL_VOLUME}}'
  CATALOG = 'SNOWFLAKE'
  BASE_LOCATION = 'balloon_lab/dt_realtime_scores'
AS
SELECT
  w.player,
  w.total_score,
  w.window_start,
  DATEADD(second, 15, w.window_start) AS window_end
FROM (
  SELECT
    e.player,
    SUM(e.score_i) AS total_score,
    TIME_SLICE(e.ts, 15, 'SECOND') AS window_start
  FROM (
    SELECT
      v:player::STRING AS player,
      v:balloon_color::STRING AS balloon_color,
      v:score::INTEGER AS score_i,
      v:favorite_color_bonus::BOOLEAN AS fav_bonus,
      v:event_ts::TIMESTAMP_NTZ AS ts
    FROM (
      SELECT PARSE_JSON(event) AS v
      FROM {{BRONZE_ICEBERG_TABLE}}
    ) q
  ) e
  GROUP BY e.player, TIME_SLICE(e.ts, 15, 'SECOND')
) w;

---

## Step 6: Balloon Colored Pops (`dt_balloon_colored_pops`)

Combines per-player, per-color breakdown **with 15-second time windows**. Gives the most granular view of who popped what color and when.

In [ ]:
%%sql -r dt_colored_pops_result
CREATE OR REPLACE DYNAMIC ICEBERG TABLE {{DB_NAME}}.silver.dt_balloon_colored_pops (
  player STRING,
  balloon_color STRING,
  balloon_pops NUMBER(38,0),
  points_by_color NUMBER(38,0),
  bonus_hits NUMBER(38,0),
  window_start TIMESTAMP_NTZ,
  window_end TIMESTAMP_NTZ
)
  TARGET_LAG = '{{TARGET_LAG}}'
  WAREHOUSE = {{WAREHOUSE}}
  EXTERNAL_VOLUME = '{{EXTERNAL_VOLUME}}'
  CATALOG = 'SNOWFLAKE'
  BASE_LOCATION = 'balloon_lab/dt_balloon_colored_pops'
AS
SELECT
  w.player,
  w.balloon_color,
  w.balloon_pops,
  w.points_by_color,
  w.bonus_hits,
  w.window_start,
  DATEADD(second, 15, w.window_start) AS window_end
FROM (
  SELECT
    e.player,
    e.balloon_color,
    COUNT(*) AS balloon_pops,
    SUM(e.score_i) AS points_by_color,
    COUNT_IF(e.fav_bonus) AS bonus_hits,
    TIME_SLICE(e.ts, 15, 'SECOND') AS window_start
  FROM (
    SELECT
      v:player::STRING AS player,
      v:balloon_color::STRING AS balloon_color,
      v:score::INTEGER AS score_i,
      v:favorite_color_bonus::BOOLEAN AS fav_bonus,
      v:event_ts::TIMESTAMP_NTZ AS ts
    FROM (
      SELECT PARSE_JSON(event) AS v
      FROM {{BRONZE_ICEBERG_TABLE}}
    ) q
  ) e
  GROUP BY e.player, e.balloon_color, TIME_SLICE(e.ts, 15, 'SECOND')
) w;

---

## Step 7: Color Performance Trends (`dt_color_performance_trends`)

Tracks **average score per pop** and **total pops by balloon color** across 15-second windows. Reveals which colors are most rewarding over time.

In [ ]:
%%sql -r dt_trends_result
CREATE OR REPLACE DYNAMIC ICEBERG TABLE {{DB_NAME}}.silver.dt_color_performance_trends (
  balloon_color STRING,
  avg_score_per_pop NUMBER(38,6),
  total_pops NUMBER(38,0),
  window_start TIMESTAMP_NTZ,
  window_end TIMESTAMP_NTZ
)
  TARGET_LAG = '{{TARGET_LAG}}'
  WAREHOUSE = {{WAREHOUSE}}
  EXTERNAL_VOLUME = '{{EXTERNAL_VOLUME}}'
  CATALOG = 'SNOWFLAKE'
  BASE_LOCATION = 'balloon_lab/dt_color_performance_trends'
AS
SELECT
  w.balloon_color,
  w.avg_score_per_pop,
  w.total_pops,
  w.window_start,
  DATEADD(second, 15, w.window_start) AS window_end
FROM (
  SELECT
    e.balloon_color,
    AVG(e.score_i) AS avg_score_per_pop,
    COUNT(*) AS total_pops,
    TIME_SLICE(e.ts, 15, 'SECOND') AS window_start
  FROM (
    SELECT
      v:player::STRING AS player,
      v:balloon_color::STRING AS balloon_color,
      v:score::INTEGER AS score_i,
      v:favorite_color_bonus::BOOLEAN AS fav_bonus,
      v:event_ts::TIMESTAMP_NTZ AS ts
    FROM (
      SELECT PARSE_JSON(event) AS v
      FROM {{BRONZE_ICEBERG_TABLE}}
    ) q
  ) e
  GROUP BY e.balloon_color, TIME_SLICE(e.ts, 15, 'SECOND')
) w;

---

## Step 8: Verify Dynamic Tables

Run the queries below to confirm all five Dynamic Iceberg Tables are visible, populated, and returning expected data. Wait for an initial refresh to complete before running these — you can check refresh status in **Snowsight → Data → Dynamic Tables** or by inspecting the `scheduling_state` column from `SHOW DYNAMIC TABLES`.

### 8a — Discovery: all five DTs visible in this schema

In [ ]:
%%sql -r dt_verify_result
SHOW DYNAMIC TABLES LIKE 'dt_%' IN {{DB_NAME}}.silver;

### 8b — Player Leaderboard: top 15 players by score

In [ ]:
%%sql -r verify_leaderboard
SELECT player, total_score, bonus_pops, last_event_ts
FROM {{DB_NAME}}.silver.dt_player_leaderboard
ORDER BY total_score DESC NULLS LAST
LIMIT 5;

In [ ]:
%%sql -r verify_leaderboard_count
SELECT COUNT(*) AS leaderboard_rows FROM {{DB_NAME}}.silver.dt_player_leaderboard;

### 8c — Balloon Color Stats: player × color breakdown

In [ ]:
%%sql -r verify_color_stats
SELECT player, balloon_color, balloon_pops, points_by_color, bonus_hits, last_event_ts
FROM {{DB_NAME}}.silver.dt_balloon_color_stats
ORDER BY player, points_by_color DESC NULLS LAST
LIMIT 20;

### 8d — Real-Time Scores: 15-second windowed scores

In [ ]:
%%sql -r verify_realtime
SELECT player, total_score, window_start, window_end
FROM {{DB_NAME}}.silver.dt_realtime_scores
ORDER BY window_start DESC, player
LIMIT 20;

In [ ]:
%%sql -r verify_realtime_spans
SELECT
  COUNT(*) AS windowed_rows,
  COUNT_IF(window_end = DATEADD(second, 15, window_start)) AS rows_with_15s_span
FROM {{DB_NAME}}.silver.dt_realtime_scores;

### 8e — Balloon Colored Pops: window × player × color

In [ ]:
%%sql -r verify_colored_pops
SELECT player, balloon_color, balloon_pops, window_start, window_end
FROM {{DB_NAME}}.silver.dt_balloon_colored_pops
ORDER BY window_start DESC, player, balloon_color
LIMIT 20;

### 8f — Color Performance Trends: avg score per pop by color over time

In [ ]:
%%sql -r verify_trends
SELECT balloon_color, avg_score_per_pop, total_pops, window_start, window_end
FROM {{DB_NAME}}.silver.dt_color_performance_trends
ORDER BY window_start DESC, balloon_color
LIMIT 20;

### 8g — Optional: Bronze vs Silver Row-Count Parity

Uncomment and update the `BRONZE_ICEBERG_TABLE` reference below to compare distinct player counts between bronze and silver. After a full DT refresh, `dt_players` should equal `bronze_distinct_players`.

In [ ]:
%%sql -r verify_parity
-- Uncomment and run after updating the bronze table FQN if needed:
/*
WITH bronze AS (
  SELECT PARSE_JSON(event):player::STRING AS player
  FROM {{BRONZE_ICEBERG_TABLE}}
)
SELECT
  (SELECT COUNT(*) FROM {{DB_NAME}}.silver.dt_player_leaderboard) AS dt_players,
  (SELECT COUNT(DISTINCT player) FROM bronze) AS bronze_distinct_players;
*/

In [ ]:
%%sql -r set_schema_result
USE SCHEMA silver;